# Developer

This final notebook is designed to give you insight into **how `executorlib` works internally** and how to **debug workflows** when things go wrong. The topics covered here are relevant when:

- Functions transfer large amounts of data between the main process and worker processes
- A few calls in a large parameter sweep fail and you need to diagnose which ones and why
- You want to test file-based (`ClusterExecutor`) behaviour locally without a scheduler
- You need to interface `executorlib` with external executables that are not Python programs

**Learning objectives:** After this notebook you will be able to:
- Measure serialisation overhead with `log_obj_size`
- Capture per-function errors in log files without crashing the workflow
- Use `TestClusterExecutor` to debug cluster-executor behaviour locally
- Wrap external command-line executables in `executorlib` functions using `subprocess`
- Build an interactive interface to a long-running external process using block allocation and `init_function`


## Data Transfer

`executorlib` communicates between the main Python process and worker processes by **serialising** (pickling) function arguments and return values. This serialisation happens every time a function is submitted and every time a result is retrieved. For most scientific data types (numpy arrays, Python scalars, ASE Atoms objects) the overhead is negligible.

However, if a function takes or returns a very large object — a multi-gigabyte array, a large graph, or a complex nested data structure — the serialisation time can dominate the total runtime. Setting `log_obj_size=True` on any executor prints the **size in bytes** of each serialised object to the console, helping you identify unexpected data transfer bottlenecks early.

> **Tip:** If a function argument is large and does not change between calls (e.g. a > pre-computed dataset), consider passing it through `init_function` instead of as a regular > argument. It is serialised once at worker startup rather than once per call.


In [1]:
from executorlib import SingleNodeExecutor

with SingleNodeExecutor(log_obj_size=True) as exe:
    future = exe.submit(sum, [1, 1])
    print(future.result())

Send dictionary of size: 101
Received dictionary of size: 59
Send dictionary of size: 69
Received dictionary of size: 58


2


## Write Log Files

When sampling a large parameter space with many function calls it is common for a small fraction of parameter combinations to cause errors — a numerical instability at the boundary of the parameter range, a missing input file for a specific structure, or a corner case in the physics. Allowing one failure to crash the entire workflow wastes the results already computed.

Setting `"error_log_file"` in `resource_dict` captures any exception raised by the function — along with the **function name** and **input parameters** — in a text file. This means:

1. The workflow continues processing remaining parameter combinations
2. Failed calls can be re-run later after debugging

The `"error_log_file"` key can be set per-function in `submit()` or globally as a default `resource_dict` in the executor constructor. The example below sets it globally via the `resource_dict` argument to `SingleNodeExecutor`.


In [2]:
from executorlib import SingleNodeExecutor

In [3]:
def my_funct(i, j): 
    if i == 2 and j == 2:
        raise ValueError()
    else: 
        return i * j + i + j

A `try` / `except` block is used here to prevent the notebook cell from stopping with an unhandled exception. In a real workflow you would not need this — the error is captured in the log file and the `Future` object for the failed call will raise the exception only when `.result()` is called on that specific future. All other futures remain unaffected.


In [4]:
with SingleNodeExecutor(resource_dict={"error_log_file": "error.out"}) as exe:
    future_lst = []
    for i in range(4):
        for j in range(4):
            future_lst.append(exe.submit(my_funct, i=i, j=j))
    try:
        print([f.result() for f in future_lst])
    except ValueError:
        pass

The log file is a plain text file that can be opened with any editor or read with Python's `open()`. The critical feature is that the log contains not just the traceback but also:
- The **function name** that raised the exception
- The **input parameters** (`kwargs: {'i': 2, 'j': 2}`) that caused the failure

This allows you to reproduce the exact failing call in an interactive session and debug it without having to re-run the entire parameter sweep.


In [5]:
with open("error.out") as f:
    content = f.readlines()

In [6]:
content

['function: <function my_funct at 0x7fd2361cf740>\n',
 'args: ()\n',
 "kwargs: {'i': 2, 'j': 2}\n",
 'Traceback (most recent call last):\n',
 '  File "/srv/conda/envs/notebook/lib/python3.13/site-packages/executorlib/backend/interactive_serial.py", line 56, in main\n',
 '    output = call_funct(input_dict=input_dict, funct=None, memory=memory)\n',
 '  File "/srv/conda/envs/notebook/lib/python3.13/site-packages/executorlib/standalone/interactive/backend.py", line 33, in call_funct\n',
 '    return funct(input_dict["fn"], *input_dict["args"], **input_dict["kwargs"])\n',
 '  File "/srv/conda/envs/notebook/lib/python3.13/site-packages/executorlib/standalone/interactive/backend.py", line 22, in funct\n',
 '    return args[0].__call__(*args[1:], **kwargs)\n',
 '           ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^\n',
 '  File "/tmp/ipykernel_2748/3167739528.py", line 3, in my_funct\n',
 'ValueError\n',
 'function: <function my_funct at 0x7fdbccaff740>\n',
 'args: ()\n',
 "kwargs: {'i': 2, 'j': 2

## TestClusterExecutor

The `SingleNodeExecutor` mimics the behaviour of `FluxJobExecutor` / `SlurmJobExecutor` (socket-based communication) but runs locally. However, the `FluxClusterExecutor` and `SlurmClusterExecutor` use a fundamentally different communication mechanism: they write serialised function arguments to HDF5 files on disk and read results back from those files. This file-based round-trip introduces potential failure modes that socket communication does not have:

- Permissions issues with the cache directory
- Version incompatibilities in the HDF5 format between executorlib releases
- Race conditions when multiple workers access the same cache simultaneously

`TestClusterExecutor` replicates the file-based communication of `FluxClusterExecutor` / `SlurmClusterExecutor` but runs entirely locally, allowing you to reproduce and debug cluster-executor-specific issues without an HPC system. It accepts the same parameters as the cluster executors.


In [7]:
from executorlib.api import TestClusterExecutor

with TestClusterExecutor(cache_directory="test") as exe:
    future = exe.submit(sum, [1, 1])
    print(future.result())

2


## External Executables

`executorlib` is not limited to Python functions — it can also orchestrate **arbitrary command-line executables** written in any language (Fortran, C++, Julia, bash, etc.). This is important at LANL because many simulation codes are legacy Fortran programs that do not have Python bindings.

Two usage patterns are supported:

1. **Subprocess** — the executable is called once per function invocation. Suitable for programs that read input from files and write output to files (the Quantum ESPRESSO workflow in the previous notebook uses this pattern).

2. **Interactive** — the executable is started once and remains running; Python communicates with it over `stdin`/`stdout` for multiple interactions. Suitable for programs with a request-response interface (REPL, interactive simulation codes, etc.).


### Subprocess

If the external executable is invoked once per task, wrap it in a Python function using the [`subprocess`](https://docs.python.org/3/library/subprocess.html) module from the standard library. This function is then submitted to `executorlib` like any other Python function — `executorlib` handles the parallelism, and the function handles the executable invocation.

In the example below the shell command `echo test` is submitted via `execute_shell_command()` to a `SingleNodeExecutor`. In a real workflow you would replace `["echo", "test"]` with the path and arguments of your simulation executable, e.g. `["/path/to/pw.x", "-in", "input.pwi"]`.

> **Security note:** Set `shell=False` (the default) whenever possible to avoid shell > injection vulnerabilities. Only set `shell=True` if you need shell features like pipes or > glob expansion, and never construct the command string from untrusted user input.


In [8]:
from executorlib import SingleNodeExecutor

In [9]:
def execute_shell_command(
    command: list, universal_newlines: bool = True, shell: bool = False
):
    import subprocess

    return subprocess.check_output(
        command, universal_newlines=universal_newlines, shell=shell
    )

In [10]:
with SingleNodeExecutor() as exe:
    future = exe.submit(
        execute_shell_command,
        ["echo", "test"],
        universal_newlines=True,
        shell=False,
    )
    print(future.result())

test



### Interactive

The more complex case is maintaining a **persistent connection** to an external executable — sending it inputs and reading back outputs multiple times without restarting it. This pattern is needed when:

- Program startup is expensive (loading large data files, JIT compilation)
- The program maintains internal state between calls
- The program implements a conversational protocol (REPL, interactive solver)

`executorlib`'s `block_allocation` feature is the key enabler: with `block_allocation=True` the worker process persists between function calls, and the `init_function` starts the external executable once when the worker initialises.

The example here uses a simple Python script `count.py` as a stand-in for any external executable — in practice this could be a Fortran REPL, a compiled solver, or any program that reads commands from `stdin` and writes results to `stdout`.


In [11]:
count_script = """\
def count(iterations):
    for i in range(int(iterations)):
        print(i)
    print("done")


if __name__ == "__main__":
    while True:
        user_input = input()
        if "shutdown" in user_input:
            break
        else:
            count(iterations=int(user_input))
"""

with open("count.py", "w") as f:
    f.writelines(count_script)

The `init_function` starts the external executable as a subprocess and returns a dictionary whose keys become keyword arguments to every subsequently submitted function. Here the subprocess is started with `stdin=PIPE` and `stdout=PIPE` so that Python can write to its standard input and read from its standard output.

The `init_function` runs **once when the worker process starts** (not once per function call), so the external executable's startup overhead is amortised across all function calls in the block allocation.


In [12]:
def init_process():
    import subprocess

    return {
        "process": subprocess.Popen(
            ["python", "count.py"],
            stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,
            universal_newlines=True,
            shell=False,
        )
    }

The `interact()` function is the core of the communication protocol. It:
1. Writes a command string to the external executable's `stdin`, followed by a newline `\n`
2. Calls `flush()` to ensure the data is sent immediately (buffering can cause deadlocks if the flush is omitted)
3. Reads the response back from `stdout` either for a fixed number of lines or until a sentinel pattern is matched

The exact implementation depends on the protocol your executable uses. Always end commands with `\n` and flush immediately.


In [13]:
def interact(shell_input, process, lines_to_read=None, stop_read_pattern=None):
    process.stdin.write(shell_input)
    process.stdin.flush()
    lines_count = 0
    output = ""
    while True:
        output_current = process.stdout.readline()
        output += output_current
        lines_count += 1
        if stop_read_pattern is not None and stop_read_pattern in output_current:
            break
        elif lines_to_read is not None and lines_to_read == lines_count:
            break
    return output

The `shutdown()` function sends the termination command to the external executable before the worker process exits. Without a graceful shutdown the subprocess might be left as a zombie process. The specific termination signal depends on the executable; here `"shutdown\n"` is the sentinel that `count.py` recognises.


In [14]:
def shutdown(process):
    process.stdin.write("shutdown\n")
    process.stdin.flush()

With these three utility functions (`init_process`, `interact`, `shutdown`) `executorlib` can communicate with any external executable that uses a `stdin`/`stdout` protocol. The `shutdown_function` parameter ensures clean termination.

This pattern is currently considered a **developer feature** — it requires knowledge of the external executable's communication protocol and careful handling of buffering, encoding, and error conditions. For production use it is recommended to wrap the executable in a more robust Python adapter class. The `FluxJobExecutor` / `SlurmJobExecutor`-based pattern from the Application notebook (calling the executable once per function via `subprocess`) is simpler and sufficient for most simulation codes.


In [15]:
with SingleNodeExecutor(
    max_workers=1,
    init_function=init_process,
    block_allocation=True,
) as exe:
    future = exe.submit(
        interact, shell_input="4\n", lines_to_read=5, stop_read_pattern=None
    )
    print(future.result())
    future_shutdown = exe.submit(shutdown)
    print(future_shutdown.result())

0
1
2
3
done

None
